# SmartFarm — AI inference on camera frames

This notebook pulls camera frames from the web-server over the shared docker
network and runs inference on them. Frames are pulled by HTTP (no SD-card
files); `ETag` / `304 Not Modified` means we only process **new** frames.

Contract: see `../../docs/ai-frame-pull.md`. Poller script: `frame_poller.py`.

In [ ]:
import io, os, time
import requests
from PIL import Image
from IPython.display import display

BASE = os.environ.get('WEB_SERVER_URL', 'http://web-server:3000')
FRAME_URL = BASE + '/api/v1/camera/frame.jpg'
STATUS_URL = BASE + '/api/v1/camera/status'
print('frame url:', FRAME_URL)

## 1. Check the camera is online, grab one frame

In [ ]:
print(requests.get(STATUS_URL, timeout=10).json())

resp = requests.get(FRAME_URL, timeout=10)
if resp.status_code == 200:
    img = Image.open(io.BytesIO(resp.content)).convert('RGB')
    print('frame seq', resp.headers.get('X-Frame-Seq'), 'size', img.size)
    display(img)
elif resp.status_code == 503:
    print('no frame yet - the camera has not pushed one')
else:
    print('unexpected', resp.status_code)

## 2. Continuous inference loop (ETag dedup)

Replace `infer()` with your model (torchvision, jetcam, etc.). The loop only
calls it when a genuinely new frame arrives. Interrupt the kernel to stop.

In [ ]:
POLL_SECONDS = float(os.environ.get('POLL_SECONDS', '5'))

def infer(img, seq):
    # TODO: run a real model here.
    print('inferred frame seq=%s size=%s' % (seq, img.size))

etag = None
try:
    while True:
        headers = {'If-None-Match': etag} if etag else {}
        r = requests.get(FRAME_URL, headers=headers, timeout=10)
        if r.status_code == 200:
            etag = r.headers.get('ETag', etag)
            img = Image.open(io.BytesIO(r.content)).convert('RGB')
            infer(img, r.headers.get('X-Frame-Seq'))
        elif r.status_code not in (304, 503):
            print('unexpected', r.status_code)
        time.sleep(POLL_SECONDS)
except KeyboardInterrupt:
    print('stopped')